# 🧪 Solving DeepMind's Alchemy Benchmark with Epiplexity-Guided Meta-Learning

**Author:** Chris Martinez 
**Date:** January 2026

## Ideas taken from these papers

Original Alchemy Paper from Deepmind:
https://arxiv.org/abs/2102.02926

Code base and ideas taken from Sony AI and Deepmind:
https://arxiv.org/abs/2112.08360

Epiplexity Paper:
https://arxiv.org/abs/2601.03220

## Overview

This notebook presents a novel approach to solving [DeepMind's Alchemy](https://github.com/deepmind/dm_alchemy) meta-reinforcement learning benchmark using **Epiplexity** as a model selection criterion.

### What is Epiplexity?

Epiplexity (from ["From Entropy to Epiplexity" - Finzi et al., 2026](https://arxiv.org/abs/2601.03220)) measures the **structural information** a computationally-bounded learner extracts from data. Unlike entropy (which captures randomness), epiplexity captures *learnable structure*.

**Key insight:** Instead of just maximizing reward, we select models that *learn more structure* from each episode.

### The Approach: Epiplexity Duel

1. Clone the current best model into two variants (A and B) with different entropy coefficients
2. Run both through the same episode environment
3. Compute epiplexity for each: `Δ = value_loss_before - value_loss_after`
4. **Keep the model with higher epiplexity** (more structural learning)
5. Repeat

This creates an evolutionary pressure toward models that extract more learnable structure from experience.

### Results

After around 1500 episodes, the model achieves roughly **200 reward per episode** on the Alchemy benchmark, demonstrating successful meta-learning of the underlying chemistry rules.

---

## 1. Installation

Run these cells to install DeepMind Alchemy. You may have to run it twice for some reason, once after restart. If in colab, you may need to use the older 2025.07 kernels:

In [ ]:
!git clone https://github.com/deepmind/dm_alchemy.git
!pip install wheel
!pip install --upgrade setuptools
!pip install ./dm_alchemy

In [ ]:
!pip uninstall protobuf
!pip install protobuf

## 2. Configuration

All hyperparameters are defined here - no external config file needed:

In [ ]:
#==============================================================================
# CONFIGURATION - All settings in one place
#==============================================================================

CONFIG = {
    # Run settings
    "run-title": "A2C_EPN_EPIPLEXITY",
    "device": "cuda",  # or "cpu"
    "seed": 42,
    
    # Paths (modify for your setup)
    "save-path": "./models",
    "log-path": "./logs",
    "save-interval": 100,
    
    # Resume training (set to path of .pt file to resume)
    "load-path": None,  # e.g., "./models/checkpoint_1000.pt"
    "start-episode": 0,
    
    # Agent hyperparameters
    "agent": {
        "lr": 1.0e-3,
        "gamma": 0.70,
        "value-loss-weight": 0.5,
        "entropy-weight": 0.1,  # Base entropy (A gets 0.5x, B gets 2x)
        "n-step-update": 20,
        "grad-clip-norm": 1.0,
        "n-workers": 8,  # Parallel environments
        
        # Architecture
        "obs-dim": 39,
        "hidden-dim": 256,
        "encoder": [32, 32],
        
        # Attention/Memory
        "attn-num-heads": 4,
        "attn-dim": 64,
        "dict-len": 150,  # Episodic memory size
        "memory-dim": 81,
    },
    
    # Task settings
    "task": {
        "n-episodes": 20000,
        "n-actions": 40,
    },
    
    # Testing
    "test": False,
    "test-episodes": 100,
}

print(f"Configuration loaded: {CONFIG['run-title']}")
print(f"Device: {CONFIG['device']}")
print(f"Workers: {CONFIG['agent']['n-workers']}")

## 3. Imports and Setup

In [ ]:
import numpy as np
import math
import os
import copy
import pickle
from datetime import datetime
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
from torch.distributions import Categorical
from torch.utils.tensorboard import SummaryWriter

# DeepMind Alchemy
from dm_alchemy import symbolic_alchemy
from dm_alchemy.encode import chemistries_proto_conversion
from dm_alchemy.types import utils

# Numpy compatibility fixes
np.object = object
np.int = int

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 4. Parallel Environment

In [ ]:
#==============================================================================
# PARALLEL ENVIRONMENT
#==============================================================================

def worker(env_seed, master_end, worker_end):
    """Worker process for parallel environment execution."""
    master_end.close()
    
    level_name = 'alchemy/perceptual_mapping_randomized_with_rotation_and_random_bottleneck'
    env = symbolic_alchemy.get_symbolic_alchemy_level(
        level_name, seed=env_seed, max_steps_per_trial=15
    )
    
    while True:
        cmd, action = worker_end.recv()
        if cmd == 'step':
            timestep = env.step(action)
            ob = timestep.observation["symbolic_obs"]
            reward = timestep.reward
            done = timestep.last()
            worker_end.send((ob, reward, done))
        elif cmd == 'reset':
            timestep = env.reset()
            ob = timestep.observation["symbolic_obs"]
            worker_end.send(ob)
        elif cmd == 'close':
            worker_end.close()
            break


class ParallelEnv:
    """Manages multiple environment workers for parallel training."""
    
    def __init__(self, n_workers, base_seed):
        self.base_seed = base_seed
        self.nenvs = n_workers
        self.waiting = False
        self.closed = False
        self.workers = []
        
        master_ends, worker_ends = zip(*[mp.Pipe() for _ in range(self.nenvs)])
        self.master_ends, self.worker_ends = master_ends, worker_ends
        
        for worker_id, (master_end, worker_end) in enumerate(zip(master_ends, worker_ends)):
            p = mp.Process(
                target=worker,
                args=((worker_id + 1) * self.base_seed, master_end, worker_end)
            )
            p.daemon = True
            p.start()
            self.workers.append(p)
        
        for worker_end in worker_ends:
            worker_end.close()
    
    def step(self, actions):
        for master_end, action in zip(self.master_ends, actions):
            master_end.send(('step', action))
        results = [master_end.recv() for master_end in self.master_ends]
        obs, rews, dones = zip(*results)
        return np.stack(obs), np.stack(rews), np.stack(dones)
    
    def reset(self):
        for master_end in self.master_ends:
            master_end.send(('reset', None))
        return np.stack([master_end.recv() for master_end in self.master_ends])
    
    def close(self):
        if self.closed:
            return
        for master_end in self.master_ends:
            master_end.send(('close', None))
        for worker in self.workers:
            worker.join()
        self.closed = True

## 5. Neural Network Architecture

In [ ]:
#==============================================================================
# EPISODIC MEMORY
#==============================================================================

class EpisodicMemory:
    """Stores (state, action, reward, done) tuples from the episode."""
    
    def __init__(self, n_workers, mem_size, mem_dim):
        self.mem_dim = mem_dim
        self.mem_size = mem_size
        self.n_workers = n_workers
        self.reset()
    
    def reset(self):
        self.pointer = 0
        self.memory = np.zeros((self.n_workers, self.mem_size, self.mem_dim))
    
    def push(self, memories):
        self.memory[:, self.pointer, :] = memories
        self.pointer += 1
    
    def generate_mask(self):
        mask = np.zeros((self.n_workers, self.mem_size))
        mask[:, :self.pointer] = 1
        return mask


#==============================================================================
# ATTENTION MODULES
#==============================================================================

class ScaledDotProductAttention(nn.Module):
    """Standard scaled dot-product attention."""
    
    def forward(self, query, key, value, softmax_dim, mask=None):
        dk = query.size()[-1]
        scores = query.matmul(key.transpose(-2, -1)) / math.sqrt(dk)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attention = F.softmax(scores, dim=softmax_dim)
        return attention.matmul(value), attention


class MultiHeadAttention(nn.Module):
    """Multi-head attention for episodic memory retrieval."""
    
    def __init__(self, dim, q_dim, k_dim, v_dim, head_num, bias=True, 
                 activation=F.relu, softmax_dim=-1):
        super().__init__()
        if dim % head_num != 0:
            raise ValueError(f'dim ({dim}) must be divisible by head_num ({head_num})')
        
        self.q_dim = q_dim
        self.k_dim = k_dim
        self.v_dim = v_dim
        self.dim = dim
        self.head_num = head_num
        self.activation = activation
        self.softmax_dim = softmax_dim
        
        self.linear_q = nn.Linear(q_dim, dim, bias)
        self.linear_k = nn.Linear(k_dim, dim, bias)
        self.linear_v = nn.Linear(v_dim, dim, bias)
        self.linear_o = nn.Linear(dim, dim, bias)
    
    def forward(self, q, k, v, mask=None):
        q, k, v = self.linear_q(q), self.linear_k(k), self.linear_v(v)
        
        if self.activation is not None:
            q = self.activation(q)
            k = self.activation(k)
            v = self.activation(v)
        
        q = self._reshape_to_batches(q)
        k = self._reshape_to_batches(k)
        v = self._reshape_to_batches(v)
        
        if mask is not None:
            mask = mask.repeat(self.head_num, 1, 1)
        
        y, attn_weights = ScaledDotProductAttention()(q, k, v, self.softmax_dim, mask)
        y = self._reshape_from_batches(y)
        attn_weights = self._reshape_from_batches(attn_weights)
        
        y = self.linear_o(y)
        if self.activation is not None:
            y = self.activation(y)
        return y, attn_weights
    
    def _reshape_to_batches(self, x):
        batch_size, seq_len, in_feature = x.size()
        sub_dim = in_feature // self.head_num
        return x.reshape(batch_size, seq_len, self.head_num, sub_dim)\
                .permute(0, 2, 1, 3)\
                .reshape(batch_size * self.head_num, seq_len, sub_dim)
    
    def _reshape_from_batches(self, x):
        batch_size, seq_len, in_feature = x.size()
        batch_size //= self.head_num
        out_dim = in_feature * self.head_num
        return x.reshape(batch_size, self.head_num, seq_len, in_feature)\
                .permute(0, 2, 1, 3)\
                .reshape(batch_size, seq_len, out_dim)


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for transformer."""
    
    def __init__(self, d_model, max_len=300):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 0:
            pe[:, 1::2] = torch.cos(position * div_term)
        else:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [ ]:
#==============================================================================
# EPN TRANSFORMER - Episodic Memory Processing
#==============================================================================

class EPNTransformer(nn.Module):
    """Transformer for processing episodic memory."""
    
    def __init__(self, config, layer_norm_size, project=True):
        super().__init__()
        self.obs_dim = config["obs-dim"]
        self.mem_size = config["dict-len"]
        self.attn_dim = config["attn-dim"]
        self.num_heads = config["attn-num-heads"]
        self.mem_dim = config["memory-dim"]
        self.project = project
        
        self.pos_enc = PositionalEncoding(self.attn_dim)
        self.attn_proj = nn.Linear(self.obs_dim + self.mem_dim, self.attn_dim)
        self.layer_norm = nn.LayerNorm((layer_norm_size, self.attn_dim))
        
        self.attention = MultiHeadAttention(
            dim=self.attn_dim,
            q_dim=self.attn_dim,
            k_dim=self.attn_dim,
            v_dim=self.attn_dim,
            head_num=self.num_heads,
        )
        
        self.shared_mlp = nn.Sequential(
            nn.Linear(self.attn_dim, self.attn_dim),
            nn.ELU(),
            nn.Linear(self.attn_dim, self.attn_dim),
            nn.ELU(),
        )
    
    def forward(self, s_t, m_t, mask_t):
        # Expand state to match memory dimensions
        s_t_expand = s_t.unsqueeze(1).expand(*m_t.shape[:-1], self.obs_dim)
        x = torch.cat([m_t, s_t_expand], dim=-1)
        
        if self.project:
            b_i = self.pos_enc(self.attn_proj(x))
        else:
            b_i = self.pos_enc(m_t)
        
        x = self.layer_norm(b_i)
        attn_out, _ = self.attention(x, x, x, mask=mask_t.unsqueeze(1))
        b_i = F.elu(b_i + attn_out)
        b_i = self.shared_mlp(b_i)
        
        return b_i


#==============================================================================
# A2C-EPN AGENT - Actor-Critic with Episodic Memory
#==============================================================================

class A2C_EPN(nn.Module):
    """Actor-Critic agent with Episodic Memory Network."""
    
    def __init__(self, config, n_actions):
        super().__init__()
        self.obs_dim = config["obs-dim"]
        self.hidden_dim = config["hidden-dim"]
        self.attn_dim = config["attn-dim"]
        self.mem_size = config["dict-len"]
        self.n_actions = n_actions
        
        # Input: encoded obs (32) + action (40) + reward (1) + done (1) + memory embedding (64)
        self.input_dim = 32 + 40 + 2 + config['attn-dim']
        
        self.transformer = EPNTransformer(config, self.mem_size)
        self.lstm = nn.LSTMCell(self.input_dim, self.hidden_dim)
        
        self.encoder = nn.Sequential(
            nn.Linear(self.obs_dim, config["encoder"][0]),
            nn.ELU(),
            nn.Linear(config["encoder"][0], config["encoder"][1]),
            nn.ELU(),
        )
        
        self.actor = nn.Linear(self.hidden_dim, n_actions)
        self.critic = nn.Linear(self.hidden_dim, 1)
    
    def forward(self, inputs):
        s_t, a_tm1, r_tm1, d_tm1, m_t, mask_t, lstm_state = inputs
        ht, ct = lstm_state
        
        # Encode observation
        feats = self.encoder(s_t)
        
        # Process episodic memory
        z_b_i = self.transformer(s_t, m_t, mask_t)
        b_i = torch.max(z_b_i, dim=1).values  # Max pool over memory
        
        # Concatenate all inputs
        x_t = torch.cat([feats, a_tm1, d_tm1, r_tm1, b_i], dim=-1)
        
        # LSTM update
        h_t, c_t = self.lstm(x_t, (ht, ct))
        
        # Actor-Critic outputs
        value_est = self.critic(h_t)
        act_logits = self.actor(h_t)
        
        sm = F.softmax(act_logits, dim=-1)
        final_actions = Categorical(sm).sample().detach()
        lsm = F.log_softmax(act_logits, dim=-1)
        log_prob = lsm.gather(1, final_actions.unsqueeze(1))
        entropy = -(lsm * sm).sum(1, keepdim=True)
        
        return value_est, final_actions, log_prob, entropy, (h_t, c_t)

## 6. Training Functions

In [ ]:
#==============================================================================
# UTILITY FUNCTIONS
#==============================================================================

def compute_target(v_final, r_lst, mask_lst, gamma):
    """Compute n-step TD targets."""
    G = v_final.reshape(-1)
    td_target = []
    
    for r, mask in zip(r_lst[::-1], mask_lst[::-1]):
        G = r + gamma * G * mask
        td_target.append(G)
    
    return torch.tensor(td_target[::-1]).float()


def save_checkpoint(model, optimizer, episode, path, stats=None):
    """Save model checkpoint."""
    checkpoint = {
        'episode': episode,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer else None,
        'stats': stats,
    }
    torch.save(checkpoint, path)
    print(f"Saved checkpoint: {path}")


def load_checkpoint(model, path, optimizer=None, device='cuda'):
    """Load model checkpoint."""
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer and checkpoint.get('optimizer_state_dict'):
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print(f"Loaded checkpoint from episode {checkpoint.get('episode', 'unknown')}")
    return checkpoint.get('episode', 0), checkpoint.get('stats', {})

In [ ]:
#==============================================================================
# EPISODE RUNNER WITH EPIPLEXITY COMPUTATION
#==============================================================================

def run_episode_with_epiplexity(model, optimizer, envs, config, entropy_coeff, device):
    """
    Run one episode and compute epiplexity.
    
    Epiplexity = Σ(value_loss_before - value_loss_after) across updates
    
    This measures how much *structural information* the model extracts
    from the episode - models that learn more have higher epiplexity.
    """
    gamma = config["agent"]["gamma"]
    val_coeff = config["agent"]["value-loss-weight"]
    grad_clip_norm = config["agent"]["grad-clip-norm"]
    update_interval = config["agent"]["n-step-update"]
    n_workers = config["agent"]["n-workers"]
    steps_per_trial = 15
    
    model.train()
    
    # Initialize episode
    states = envs.reset()
    s_prime = states
    
    episodic_memory = EpisodicMemory(
        n_workers,
        config["agent"]["dict-len"],
        config["agent"]["memory-dim"]
    )
    
    p_action = np.zeros((n_workers, 40))
    p_reward = np.zeros((n_workers, 1))
    p_done = np.zeros((n_workers, 1))
    done = np.array([True] * n_workers)
    
    episode_rewards = np.zeros(n_workers)
    steps_this_trial = 1
    
    ht = torch.zeros((n_workers, model.hidden_dim), device=device)
    ct = torch.zeros((n_workers, model.hidden_dim), device=device)
    
    episode_epiplexity = 0.0
    update_counter = 0
    
    # Episode loop
    while True:
        r_lst, mask_lst = [], []
        values, log_probs, entropies = [], [], []
        cached_inputs = []
        
        # Reset LSTM state at episode boundaries
        if all(done):
            ht = torch.zeros((n_workers, model.hidden_dim), device=device)
            ct = torch.zeros((n_workers, model.hidden_dim), device=device)
        else:
            ht, ct = ht.detach(), ct.detach()
        
        # Collect rollout
        for _ in range(update_interval):
            mem_mask = episodic_memory.generate_mask()
            
            # Cache inputs for post-update evaluation
            cached_inputs.append((
                torch.from_numpy(states).float().to(device),
                torch.from_numpy(p_action).float().to(device),
                torch.from_numpy(p_reward).float().to(device),
                torch.from_numpy(p_done).float().to(device),
                torch.from_numpy(episodic_memory.memory).float().to(device),
                torch.from_numpy(mem_mask).float().to(device),
            ))
            
            # Forward pass
            value_est, selected_actions, log_prob, entropy, (ht, ct) = model((
                torch.from_numpy(states).float().to(device),
                torch.from_numpy(p_action).float().to(device),
                torch.from_numpy(p_reward).float().to(device),
                torch.from_numpy(p_done).float().to(device),
                torch.from_numpy(episodic_memory.memory).float().to(device),
                torch.from_numpy(mem_mask).float().to(device),
                (ht, ct)
            ))
            
            actions = selected_actions.cpu().numpy()
            s_prime, rewards, done = envs.step(actions)
            episode_rewards += rewards
            
            # Trial boundary handling
            if steps_this_trial == steps_per_trial:
                steps_this_trial = 1
                p_action[:] = 0
                p_reward[:] = 0
                p_done[:] = 0
                states = s_prime
            elif not all(done):
                # Store in episodic memory
                memories = np.concatenate([
                    states,
                    np.eye(40)[actions],
                    rewards[:, np.newaxis],
                    done[:, np.newaxis]
                ], axis=-1)
                episodic_memory.push(memories)
                
                p_reward = rewards[:, np.newaxis]
                p_action = np.eye(40)[actions]
                p_done = done[:, np.newaxis]
                states = s_prime
                steps_this_trial += 1
            
            r_lst.append(rewards)
            mask_lst.append(1 - done)
            values.append(value_est)
            log_probs.append(log_prob)
            entropies.append(entropy)
            
            if all(done):
                break
        
        # Bootstrap value
        mem_mask = episodic_memory.generate_mask()
        value_estimate, _, _, _, _ = model((
            torch.from_numpy(s_prime).float().to(device),
            torch.from_numpy(np.eye(40)[actions]).float().to(device),
            torch.from_numpy(rewards[:, np.newaxis]).float().to(device),
            torch.from_numpy(done[:, np.newaxis]).float().to(device),
            torch.from_numpy(episodic_memory.memory).float().to(device),
            torch.from_numpy(mem_mask).float().to(device),
            (ht, ct)
        ))
        
        values = torch.stack(values)
        log_probs = torch.stack(log_probs)
        entropies = torch.stack(entropies)
        
        td_target = compute_target(
            value_estimate.detach().cpu().numpy(),
            r_lst, mask_lst, gamma
        ).to(device)
        
        td_target_vec = td_target.reshape(-1)
        values_vec = values.reshape(-1)
        advantage = td_target_vec - values_vec
        
        # ═══════════════════════════════════════════════════════════════
        # EPIPLEXITY COMPUTATION
        # ═══════════════════════════════════════════════════════════════
        # Value loss BEFORE gradient update
        value_loss_before = F.smooth_l1_loss(values_vec.detach(), td_target_vec).item()
        
        # Standard A2C losses
        value_loss = F.smooth_l1_loss(values_vec, td_target_vec)
        action_loss = -(log_probs.reshape(-1) * advantage.detach()).mean()
        entropy_mean = entropies.reshape(-1).mean()
        
        loss = (value_loss * val_coeff) + action_loss - (entropy_mean * entropy_coeff)
        
        # Gradient update
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        optimizer.step()
        
        # Value loss AFTER gradient update (re-evaluate same inputs)
        with torch.no_grad():
            values_after = []
            ht_eval, ct_eval = ht.detach(), ct.detach()
            
            for (s, pa, pr, pd, mem, mm) in cached_inputs:
                v, _, _, _, (ht_eval, ct_eval) = model((
                    s, pa, pr, pd, mem, mm, (ht_eval, ct_eval)
                ))
                values_after.append(v)
            
            values_after = torch.stack(values_after).reshape(-1)
            value_loss_after = F.smooth_l1_loss(values_after, td_target_vec).item()
        
        # Epiplexity = improvement in value prediction
        # Higher = more structural information extracted
        epi_delta = value_loss_before - value_loss_after
        episode_epiplexity += epi_delta
        update_counter += 1
        
        if all(done):
            break
    
    return {
        "reward_mean": float(episode_rewards.mean()),
        "reward_sum": float(episode_rewards.sum()),
        "episode_epiplexity": float(episode_epiplexity),
        "updates_in_episode": int(update_counter),
    }

## 7. Main Training Loop - Epiplexity Duel

In [ ]:
#==============================================================================
# MAIN TRAINING LOOP - EPIPLEXITY DUEL
#==============================================================================

def train(config):
    """
    Main training loop using Epiplexity Duel:
    
    1. Clone model into A (low entropy) and B (high entropy)
    2. Run both through episode
    3. Keep model with higher epiplexity
    4. Repeat
    
    This creates evolutionary pressure toward models that
    extract more structural information from experience.
    """
    device = config["device"]
    n_actions = config["task"]["n-actions"]
    n_episodes = config["task"]["n-episodes"]
    learning_rate = config["agent"]["lr"]
    entropy_base = config["agent"]["entropy-weight"]
    save_interval = config["save-interval"]
    
    # Create directories
    os.makedirs(config["save-path"], exist_ok=True)
    os.makedirs(config["log-path"], exist_ok=True)
    
    # TensorBoard
    writer = SummaryWriter(
        log_dir=os.path.join(config["log-path"], config["run-title"])
    )
    
    # Initialize model
    base_model = A2C_EPN(config["agent"], n_actions).to(device)
    
    # Load checkpoint if specified
    start_episode = config["start-episode"]
    if config.get("load-path") and os.path.exists(config["load-path"]):
        start_episode, _ = load_checkpoint(base_model, config["load-path"], device=device)
    
    # Entropy coefficients for duel
    entropy_low = entropy_base * 0.5
    entropy_high = entropy_base * 2.0
    
    wins_A = 0
    wins_B = 0
    
    print("="*60)
    print(f"Starting Epiplexity Duel Training: {config['run-title']}")
    print(f"Device: {device}")
    print(f"Episodes: {n_episodes}")
    print(f"Entropy (low/high): {entropy_low:.3f} / {entropy_high:.3f}")
    print("="*60)
    
    progress = tqdm(range(start_episode, n_episodes), initial=start_episode, total=n_episodes)
    
    for epi in progress:
        # Clone models for duel
        model_A = copy.deepcopy(base_model)
        model_B = copy.deepcopy(base_model)
        
        optim_A = optim.AdamW(model_A.parameters(), lr=learning_rate)
        optim_B = optim.AdamW(model_B.parameters(), lr=learning_rate)
        
        # Create parallel environments (same seed for fair comparison)
        env_A = ParallelEnv(config["agent"]["n-workers"], 42)
        env_B = ParallelEnv(config["agent"]["n-workers"], 42)
        
        # Run episodes
        stats_A = run_episode_with_epiplexity(
            model_A, optim_A, env_A, config, entropy_low, device
        )
        stats_B = run_episode_with_epiplexity(
            model_B, optim_B, env_B, config, entropy_high, device
        )
        
        # ═══════════════════════════════════════════════════════════════
        # SELECT WINNER BY EPIPLEXITY (not reward!)
        # ═══════════════════════════════════════════════════════════════
        if stats_A["episode_epiplexity"] > stats_B["episode_epiplexity"]:
            base_model.load_state_dict(model_A.state_dict())
            wins_A += 1
            winner = "A (low entropy)"
        else:
            base_model.load_state_dict(model_B.state_dict())
            wins_B += 1
            winner = "B (high entropy)"
        
        # Logging
        writer.add_scalar("epi/epiplexity_A", stats_A["episode_epiplexity"], epi)
        writer.add_scalar("epi/epiplexity_B", stats_B["episode_epiplexity"], epi)
        writer.add_scalar("perf/reward_A", stats_A["reward_mean"], epi)
        writer.add_scalar("perf/reward_B", stats_B["reward_mean"], epi)
        writer.add_scalar("meta/wins_A", wins_A, epi)
        writer.add_scalar("meta/wins_B", wins_B, epi)
        
        print(
            f"[Ep {epi}] Winner: {winner} | "
            f"Epi A={stats_A['episode_epiplexity']:.4f} B={stats_B['episode_epiplexity']:.4f} | "
            f"Rew A={stats_A['reward_mean']:.1f} B={stats_B['reward_mean']:.1f} | "
            f"Wins {wins_A}/{wins_B}"
        )
        
        # Save checkpoint
        if (epi + 1) % save_interval == 0:
            save_path = os.path.join(
                config["save-path"], 
                f"{config['run-title']}_{epi+1:04d}.pt"
            )
            save_checkpoint(
                base_model, None, epi + 1, save_path,
                stats={'wins_A': wins_A, 'wins_B': wins_B}
            )
        
        env_A.close()
        env_B.close()
    
    writer.close()
    
    # Final save
    final_path = os.path.join(config["save-path"], f"{config['run-title']}_final.pt")
    save_checkpoint(base_model, None, n_episodes, final_path, 
                   stats={'wins_A': wins_A, 'wins_B': wins_B})
    
    print("\n" + "="*60)
    print("Training Complete!")
    print(f"Final wins: A={wins_A}, B={wins_B}")
    print(f"Model saved to: {final_path}")
    print("="*60)
    
    return base_model

## 8. Testing

In [ ]:
#==============================================================================
# TESTING FUNCTION
#==============================================================================

def test_model(model, config, n_episodes=100):
    """
    Test trained model on held-out episodes.
    Uses fixed chemistries for reproducible evaluation.
    """
    chems = chemistries_proto_conversion.load_chemistries_and_items(
        'chemistries/perceptual_mapping_randomized_with_random_bottleneck/chemistries'
    )
    
    device = config["device"]
    mem_dim = config["agent"]["memory-dim"]
    
    episodic_memory = EpisodicMemory(1, model.mem_size, mem_dim)
    episode_rewards = []
    
    model.eval()
    
    for ep in tqdm(range(n_episodes), desc="Testing"):
        chem, items = chems[ep]
        env = symbolic_alchemy.get_symbolic_alchemy_fixed(
            chemistry=chem, 
            episode_items=items,
            see_chemistries={
                'input_chem': utils.ChemistrySeen(content=utils.ElementContent.GROUND_TRUTH)
            }, 
            max_steps_per_trial=15
        )
        
        ct = torch.zeros((1, model.hidden_dim)).float().to(device)
        ht = torch.zeros((1, model.hidden_dim)).float().to(device)
        
        p_action = np.zeros((1, 40))
        p_reward = np.zeros((1, 1))
        p_done = np.zeros((1, 1))
        
        episode_reward = 0
        episodic_memory.reset()
        
        timestep = env.reset()
        state = timestep.observation["symbolic_obs"]
        
        while not timestep.last():
            mem_mask = episodic_memory.generate_mask()
            states = np.array([state])
            
            with torch.no_grad():
                value_est, selected_actions, log_prob, entropy, (ht, ct) = model((
                    torch.from_numpy(states).float().to(device),
                    torch.from_numpy(p_action).float().to(device),
                    torch.from_numpy(p_reward).float().to(device),
                    torch.from_numpy(p_done).float().to(device),
                    torch.from_numpy(episodic_memory.memory).float().to(device),
                    torch.from_numpy(mem_mask).float().to(device),
                    (ht, ct)
                ))
            
            action = selected_actions.cpu().numpy()
            timestep = env.step(action.item())
            done = timestep.last()
            s_prime = timestep.observation["symbolic_obs"]
            episode_reward += timestep.reward
            
            if env.is_new_trial():
                p_action = np.zeros((1, 40))
                p_reward = np.zeros((1, 1))
                p_done = np.zeros((1, 1))
                state = s_prime
            elif not done:
                memories = np.concatenate([
                    states.flatten(),
                    np.eye(40)[action[0]],
                    np.array([timestep.reward]),
                    np.array([done])
                ], axis=-1)
                episodic_memory.push(memories.reshape(1, -1))
                
                p_action = np.array([np.eye(40)[int(action)]])
                p_reward = np.array([[timestep.reward]])
                p_done = np.array([[done]])
                state = s_prime
        
        episode_rewards.append(episode_reward)
        env.close()
    
    episode_rewards = np.array(episode_rewards)
    
    print("\n" + "="*60)
    print("TEST RESULTS")
    print("="*60)
    print(f"Episodes: {n_episodes}")
    print(f"Mean Reward: {episode_rewards.mean():.2f} ± {episode_rewards.std():.2f}")
    print(f"Min/Max: {episode_rewards.min():.0f} / {episode_rewards.max():.0f}")
    print("="*60)
    
    return episode_rewards

## 9. Run Training

In [ ]:
#==============================================================================
# RUN TRAINING
#==============================================================================

if __name__ == "__main__" or True:  # Run in notebook
    
    # For testing mode, load a trained model and evaluate
    if CONFIG["test"]:
        model = A2C_EPN(CONFIG["agent"], CONFIG["task"]["n-actions"]).to(CONFIG["device"])
        load_checkpoint(model, CONFIG["load-path"], device=CONFIG["device"])
        test_model(model, CONFIG, n_episodes=CONFIG["test-episodes"])
    else:
        # Train
        trained_model = train(CONFIG)

## 10. Test Trained Model

In [ ]:
# After training, test the model:
# test_rewards = test_model(trained_model, CONFIG, n_episodes=100)

---

## How Epiplexity Works in This Implementation

The key insight from [Finzi et al., 2026](https://arxiv.org/abs/2601.03220) is that **epiplexity** measures *learnable structure* rather than randomness.

### Mathematical Definition

In the paper, epiplexity is defined as the description length of the optimal model:

```
S_T(X) = |P*|  where  P* = argmin_{P ∈ P_T} { |P| + E[log(1/P(X))] }
```

### Practical Approximation (This Implementation)

We approximate epiplexity as the **area under the loss curve above the final loss** - or equivalently, the cumulative improvement in value prediction:

```python
epiplexity = Σ (value_loss_before[t] - value_loss_after[t])
```

This captures how much *structural information* the model extracts during learning:
- **High epiplexity**: Model learned meaningful patterns
- **Low epiplexity**: Model learned little (data was random or already known)

### Why It Works for Meta-RL

In Alchemy, the agent must discover the hidden chemistry rules within each episode. By selecting models that maximize epiplexity:

1. We favor models that **extract transferable structure** from experience
2. We avoid models that **overfit to episode-specific randomness**
3. The evolutionary pressure creates meta-learning: learning *how to learn*

---

## Citation

If you use this code, please cite:

```bibtex
@article{finzi2026entropy,
  title={From Entropy to Epiplexity: Rethinking Information for Computationally Bounded Intelligence},
  author={Finzi, Marc and Qiu, Shikai and Jiang, Yiding and Izmailov, Pavel and Kolter, J Zico and Wilson, Andrew Gordon},
  journal={arXiv preprint arXiv:2601.03220},
  year={2026}
}
```